In [27]:
##!pip install "transformers[torch]"

In [28]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration


In [29]:
train_data=pd.read_csv("samsum-train.csv")
val_data=pd.read_csv("samsum-validation.csv")

In [30]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [31]:
val_data.head()

,id,dialogue,summary
0,13817023,"A: Hi Tom, are you busy tomorrow’s afternoon?\...",A will go to the animal shelter tomorrow to ge...
1,13716628,Emma: I’ve just fallen in love with this adven...,Emma and Rob love the advent calendar. Lauren ...
2,13829420,Jackie: Madison is pregnant\r\nJackie: but she...,Madison is pregnant but she doesn't want to ta...
3,13819648,Marla: <file_photo>\r\nMarla: look what I foun...,Marla found a pair of boxers under her bed.
4,13728448,Robert: Hey give me the address of this music ...,Robert wants Fred to send him the address of t...


In [32]:
train_data["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [33]:
train_data.shape

(14732, 3)

In [34]:
val_data.shape

(818, 3)

In [35]:
train_data=train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data=val_data.sample(n=500, random_state=42).reset_index(drop=True)


### Preprocessing

In [36]:
import re

In [37]:
def clean_data(text):
    text=re.sub(r"\r\n", " ", text)
    text=re.sub(r"\s+", " ", text)
    text=re.sub(r"<.*?>", " ", text)
    text=text.strip().lower()
    return text

In [38]:
train_data["dialogue"]=train_data["dialogue"].apply(clean_data)
train_data["summary"]=train_data["summary"].apply(clean_data)

val_data["dialogue"]=val_data["dialogue"].apply(clean_data)
val_data["summary"]=val_data["summary"].apply(clean_data)


In [39]:
train_data["dialogue"][0]

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet:   claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

### Tokenize

In [40]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [41]:
def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True)
    targets = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True)

    inputs["labels"] = targets["input_ids"] # token ids => add to input as labels
    return inputs

In [42]:
train_dataset=train_data.apply(tokenize, axis=1).tolist()
val_dataset=val_data.apply(tokenize, axis=1).tolist()

#### working with mmodel

In [43]:
#NLP->Generation Task
model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [44]:
import torch

In [45]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device=torch.device("cpu")
print("device: ", device)
model.to(device)
    
    

device:  cpu


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

### Training Arguments

In [46]:
training_args = TrainingArguments(
    output_dir = "./results",
    num_train_epochs = 6,
    weight_decay=0.01,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500
)

In [47]:
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [48]:
trainer.train()

C:\Users\PV\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [49]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model\\tokenizer_config.json',
 './saved_summary_model\\tokenizer.json')

In [50]:
model=T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer=T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

### Test The Core Logic for Summerization

In [51]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue)

    # tokenize
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    # generate the summary => token ids
    model.to(device)
    targets = model.generate(
        input_ids=inputs["input_ids"].to(device),
        attention_mask=inputs["attention_mask"].to(device),
        max_length=150,
        num_beams=4,
        early_stopping=True
    )

    # decode our output
    summary = tokenizer.decode(targets[0], skip_special_tokens=True)
    return summary

In [52]:
test_dialogue = """
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance.
Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experience.
Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks.
Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness is crucial.
Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies responsibly.
Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, act like black boxes.
Reporter: Experts also highlight the importance of responsible AI development, including data privacy and security.
Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be essential to harness AI's full potential while mitigating risks.
"""

summary = summarize_dialogue(test_dialogue)

print("Summary: ", summary)

Summary:  ai systems are becoming more capable due to advances in deep learning and access to large datasets. experts highlight the importance of responsible ai development, including data privacy and security.
